# Polynomial Trajectory Regression — Pixel Space vs. AE Latent Space
### Perceptual Straightening Hypothesis (PSH) · Inference-Time Smoothing

Inspired by *Rahimi & Tekalp (arXiv 2510.25420)*, this notebook compares
**polynomial trajectory regression** with 2 / 3 / 4 / 5 parameters (degree 1–4)
applied directly in **pixel space** and in the latent space of a trained
**Convolutional Autoencoder (AE)**, then analyses all methods through the
*Perceptual Straightening Hypothesis* lens.

| Experiment | Description |
|---|---|
| `pixel_poly_Np` | Poly regression (N params) directly on raw frames |
| `ae_poly_Np` | Poly regression on AE-encoded latents, then decoded |
| V1 space | Lightweight DoG + orientation-bank perceptual space |
| Metrics | PSNR, Temporal-SSIM, V1-Curvature, Straightness |


In [ ]:
# ==========================================
# CELL 1 — Config & Imports
# ==========================================
import os, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

CONFIG = {
    "device":      "cuda" if torch.cuda.is_available() else "cpu",
    "num_frames":  16,
    "img_size":    64,          # 64×64 RGB
    "poly_params": [2, 3, 4, 5],  # n coefficients → degree = n-1
    "latent_dim":  128,
    "ae_epochs":   100,
    "ae_lr":       1e-3,
    "noise_level": 0.22,
    "gif_fps":     4,
    "seed":        42,
    "save_dir":    "./experiment-results-comp",
}

DIRS = {k: os.path.join(CONFIG["save_dir"], k)
        for k in ["gifs", "plots", "tables"]}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

torch.manual_seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
DEV = CONFIG["device"]
print(f"Device: {DEV} | Frames: {CONFIG['num_frames']} | ImgSize: {CONFIG['img_size']}")


In [ ]:
# ==========================================
# CELL 2 — Synthetic Video & Utilities
# ==========================================

# ── Synthetic generator ───────────────────────────────────────────────────────
def make_sequence(num_frames, img_size, noise_level=0.22, seed=42):
    """Two Gaussian blobs with smooth motion + additive temporal noise."""
    rng = np.random.default_rng(seed)
    t_vals = np.linspace(0, 1, num_frames)
    yi, xi = np.meshgrid(
        np.linspace(0, 1, img_size),
        np.linspace(0, 1, img_size),
        indexing="ij",
    )
    clean, noisy = [], []
    for t in t_vals:
        cx1 = 0.5 + 0.25 * np.cos(2 * np.pi * t)
        cy1 = 0.5 + 0.25 * np.sin(2 * np.pi * t)
        cx2 = 0.15 + 0.70 * t
        cy2 = 0.85 - 0.70 * t
        d1  = (xi - cx1) ** 2 + (yi - cy1) ** 2
        d2  = (xi - cx2) ** 2 + (yi - cy2) ** 2
        r   = np.exp(-d1 / 0.024)
        g   = np.exp(-d2 / 0.016)
        b   = 0.45 * np.exp(-d1 / 0.04) + 0.45 * np.exp(-d2 / 0.03)
        frame = np.clip(np.stack([r, g, b], axis=-1).astype(np.float32), 0, 1)
        clean.append(frame)
        noise = rng.normal(0, noise_level, frame.shape).astype(np.float32)
        noisy.append(np.clip(frame + noise, 0, 1))
    return np.stack(noisy), np.stack(clean)   # (N, H, W, 3) float32

# ── GIF saver ─────────────────────────────────────────────────────────────────
def frames_to_gif(frames_float, path, fps=4):
    """Save (N, H, W, 3) float32 [0,1] as animated GIF."""
    imgs = [Image.fromarray(np.clip(f * 255, 0, 255).astype(np.uint8))
            for f in frames_float]
    imgs[0].save(path, save_all=True, append_images=imgs[1:],
                 duration=int(1000 / fps), loop=0)

# ── Polynomial regression ─────────────────────────────────────────────────────
def poly_regression(frames_float, n_params):
    """Fit polynomial of degree (n_params-1) to each pixel dimension independently.
    frames_float: (N, H, W, C) float32 → returns same shape, clipped to [0,1].
    Uses float64 internally for numerical stability via least-squares.
    """
    N  = frames_float.shape[0]
    t  = np.linspace(0, 1, N)
    A  = np.column_stack([t ** i for i in range(n_params)])         # (N, n_params)
    flat = frames_float.reshape(N, -1).astype(np.float64)
    coeffs, _, _, _ = np.linalg.lstsq(A, flat, rcond=None)          # (n_params, D)
    smoothed = (A @ coeffs).astype(np.float32)
    return np.clip(smoothed.reshape(frames_float.shape), 0, 1)

def poly_regression_latent(Z, n_params):
    """Same as above but for latent vectors. Z: (N, D) float32 → (N, D) float32."""
    N  = Z.shape[0]
    t  = np.linspace(0, 1, N)
    A  = np.column_stack([t ** i for i in range(n_params)])
    coeffs, _, _, _ = np.linalg.lstsq(A, Z.astype(np.float64), rcond=None)
    return (A @ coeffs).astype(np.float32)

# ── Metrics ───────────────────────────────────────────────────────────────────
def pca_2d(features):
    c  = features - features.mean(axis=0, keepdims=True)
    _, _, vh = np.linalg.svd(c.astype(np.float64), full_matrices=False)
    return (c.astype(np.float64) @ vh[:2].T).astype(np.float32)

def curvature_score(traj):
    """Mean curvature (rad) of trajectory. Lower = straighter."""
    if len(traj) < 3:
        return float("nan")
    d   = np.diff(traj.astype(np.float64), axis=0)
    dn  = d / (np.linalg.norm(d, axis=1, keepdims=True) + 1e-8)
    cos = np.clip((dn[:-1] * dn[1:]).sum(axis=1), -1 + 1e-7, 1 - 1e-7)
    return float(np.mean(np.arccos(cos)))

def straightness_score(traj):
    """Mean cosine similarity of consecutive displacement vectors. Higher = straighter."""
    if len(traj) < 3:
        return float("nan")
    d  = np.diff(traj.astype(np.float64), axis=0)
    dn = d / (np.linalg.norm(d, axis=1, keepdims=True) + 1e-8)
    return float(np.mean((dn[:-1] * dn[1:]).sum(axis=1)))

def temporal_ssim(frames_float):
    """Frame-to-frame SSIM approximation (no skimage dependency)."""
    vals = []
    for i in range(len(frames_float) - 1):
        a, b = frames_float[i].astype(np.float64), frames_float[i + 1].astype(np.float64)
        mu_a, mu_b = a.mean(), b.mean()
        var_a, var_b = a.var(), b.var()
        cov = ((a - mu_a) * (b - mu_b)).mean()
        c1, c2 = 1e-4, 9e-4
        num = (2 * mu_a * mu_b + c1) * (2 * cov + c2)
        den = (mu_a ** 2 + mu_b ** 2 + c1) * (var_a + var_b + c2) + 1e-14
        vals.append(num / den)
    return float(np.mean(vals))

def mse_psnr(pred, ref):
    mse = float(np.mean((pred.astype(np.float64) - ref.astype(np.float64)) ** 2))
    return mse, float(10 * np.log10(1.0 / (mse + 1e-10)))

# ── Generate data ─────────────────────────────────────────────────────────────
noisy_frames, clean_frames = make_sequence(
    CONFIG["num_frames"], CONFIG["img_size"],
    CONFIG["noise_level"], CONFIG["seed"]
)
print(f"Sequence: {noisy_frames.shape}, dtype={noisy_frames.dtype}, "
      f"range=[{noisy_frames.min():.2f}, {noisy_frames.max():.2f}]")


In [ ]:
# ==========================================
# CELL 3 — V1 Perceptual Space
# ==========================================

class V1PerceptualSpace(nn.Module):
    """
    Lightweight approximation of the V1 perceptual model from the paper.
    Stage 1 — RetinaND: difference-of-Gaussians center-surround filter.
    Stage 2 — V1: fixed orientation-selective filter bank.
    Both stages have frozen weights (no training).
    """
    def __init__(self, out_dim=64):
        super().__init__()
        self.retina = nn.Conv2d(3, 3, kernel_size=9, padding=4,
                                bias=False, groups=3)
        self.v1     = nn.Conv2d(3, out_dim, kernel_size=7, padding=3, bias=False)
        with torch.no_grad():
            nn.init.normal_(self.retina.weight, std=0.12)
            nn.init.normal_(self.v1.weight,     std=0.08)
        for p in self.parameters():
            p.requires_grad_(False)

    def forward(self, x):           # x: (N, 3, H, W) float32
        r        = F.relu(self.retina(x))
        surround = F.avg_pool2d(r, 5, stride=1, padding=2)
        retina   = F.relu(r - 0.4 * surround)              # DoG
        v1_out   = F.relu(self.v1(retina))
        return F.adaptive_avg_pool2d(v1_out, 1).flatten(1) # (N, out_dim)

v1_space = V1PerceptualSpace(out_dim=64).to(DEV).float()

def get_v1_features(frames_float):
    """frames_float: (N, H, W, 3) float32 numpy → (N, 64) numpy."""
    t = torch.from_numpy(frames_float).permute(0, 3, 1, 2).float().to(DEV)
    with torch.no_grad():
        return v1_space(t).cpu().numpy()

print("V1 perceptual space ready. Output dim:", 64)


In [ ]:
# ==========================================
# CELL 4 — Convolutional Autoencoder
# ==========================================

class ConvAE(nn.Module):
    """
    Lightweight ConvAE for 64×64 RGB frames.
    Encoder: 3 strided Conv2d → linear bottleneck (latent_dim).
    Decoder: linear → 3 transposed Conv2d → Sigmoid.
    """
    def __init__(self, latent_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3,  32, 3, stride=2, padding=1),   # 32×32
            nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),   # 16×16
            nn.GELU(),
            nn.Conv2d(64,128, 3, stride=2, padding=1),   # 8×8
            nn.GELU(),
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, latent_dim),
        )
        self.dec_fc = nn.Sequential(
            nn.Linear(latent_dim, 128 * 8 * 8), nn.GELU()
        )
        self.dec_conv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # 16×16
            nn.GELU(),
            nn.ConvTranspose2d(64,  32, 4, stride=2, padding=1),  # 32×32
            nn.GELU(),
            nn.ConvTranspose2d(32,   3, 4, stride=2, padding=1),  # 64×64
            nn.Sigmoid(),
        )

    def encode(self, x):            # (N, 3, H, W) → (N, latent_dim)
        return self.encoder(x)

    def decode(self, z):            # (N, latent_dim) → (N, 3, H, W)
        return self.dec_conv(self.dec_fc(z).view(-1, 128, 8, 8))

    def forward(self, x):
        return self.decode(self.encode(x))


def train_ae(frames_float, epochs=100, lr=1e-3):
    """Train AE on frames_float: (N, H, W, 3) numpy."""
    ae   = ConvAE(CONFIG["latent_dim"]).to(DEV)
    opt  = torch.optim.Adam(ae.parameters(), lr=lr)
    data = torch.from_numpy(frames_float).permute(0, 3, 1, 2).float().to(DEV)
    losses = []
    for epoch in range(epochs):
        ae.train()
        opt.zero_grad()
        loss = F.mse_loss(ae(data), data)
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if (epoch + 1) % 25 == 0:
            print(f"  Epoch {epoch+1:>3}/{epochs} | MSE loss: {loss.item():.5f}")
    ae.eval()
    return ae, losses


def encode_frames(ae, frames_float):
    t = torch.from_numpy(frames_float).permute(0, 3, 1, 2).float().to(DEV)
    with torch.no_grad():
        return ae.encode(t).cpu().numpy()   # (N, latent_dim) float32

def decode_latents(ae, Z_np):
    t = torch.from_numpy(Z_np.astype(np.float32)).to(DEV)
    with torch.no_grad():
        out = ae.decode(t).cpu().numpy()    # (N, 3, H, W)
    return np.clip(out.transpose(0, 2, 3, 1), 0, 1)  # (N, H, W, 3)


# Train on clean + noisy frames for a robust latent space
all_train = np.concatenate([clean_frames, noisy_frames], axis=0)
print("Training AE on", len(all_train), "frames (clean + noisy)...")
ae_model, ae_losses = train_ae(all_train, CONFIG["ae_epochs"], CONFIG["ae_lr"])
print("AE training complete.")


In [ ]:
# ==========================================
# CELL 5 — Experiments: Pixel & AE Latent
# ==========================================
DEGREES = [n - 1 for n in CONFIG["poly_params"]]

# ── Pixel-space polynomial regression ─────────────────────────────────────────
print("── Pixel-Space Polynomial Regression ──")
px_results = {}

frames_to_gif(noisy_frames, os.path.join(DIRS["gifs"], "noisy_baseline.gif"), CONFIG["gif_fps"])
frames_to_gif(clean_frames, os.path.join(DIRS["gifs"], "clean_reference.gif"), CONFIG["gif_fps"])
px_results["noisy_baseline"]  = {"frames": noisy_frames, "label": "Noisy Baseline"}
px_results["clean_reference"] = {"frames": clean_frames, "label": "Clean Reference"}

for n in CONFIG["poly_params"]:
    key = f"pixel_poly_{n}p"
    smoothed = poly_regression(noisy_frames, n)
    frames_to_gif(smoothed, os.path.join(DIRS["gifs"], f"{key}.gif"), CONFIG["gif_fps"])
    px_results[key] = {"frames": smoothed, "label": f"Pixel deg-{n-1} ({n}p)", "n_params": n}
    print(f"  n_params={n}  (degree {n-1})  saved.")

# ── AE latent-space polynomial regression ─────────────────────────────────────
print("\n── AE Latent-Space Polynomial Regression ──")
ae_results = {}

Z_noisy = encode_frames(ae_model, noisy_frames)
print(f"Latent: {Z_noisy.shape}, dtype={Z_noisy.dtype}, "
      f"range=[{Z_noisy.min():.2f}, {Z_noisy.max():.2f}]")

# AE reconstruction baseline (no polynomial smoothing)
recon = decode_latents(ae_model, Z_noisy)
frames_to_gif(recon, os.path.join(DIRS["gifs"], "ae_reconstruction.gif"), CONFIG["gif_fps"])
ae_results["ae_reconstruction"] = {"frames": recon, "Z": Z_noisy, "label": "AE Recon (no poly)"}

for n in CONFIG["poly_params"]:
    key      = f"ae_poly_{n}p"
    Z_smooth = poly_regression_latent(Z_noisy, n)
    dec      = decode_latents(ae_model, Z_smooth)
    frames_to_gif(dec, os.path.join(DIRS["gifs"], f"{key}.gif"), CONFIG["gif_fps"])
    ae_results[key] = {"frames": dec, "Z": Z_smooth,
                       "label": f"AE-Latent deg-{n-1} ({n}p)", "n_params": n}
    print(f"  n_params={n}  (degree {n-1})  saved.")

print("\nAll GIFs saved to:", DIRS["gifs"])


In [ ]:
# ==========================================
# CELL 6 — Metrics, Table & Plots
# ==========================================

def compute_metrics(frames_float, clean_ref, Z=None):
    mse_v, psnr_v = mse_psnr(frames_float, clean_ref)
    px_flat  = frames_float.reshape(len(frames_float), -1)
    v1_feats = get_v1_features(frames_float)
    m = {
        "mse":               mse_v,
        "psnr":              psnr_v,
        "temporal_ssim":     temporal_ssim(frames_float),
        "pixel_curvature":   curvature_score(px_flat),
        "pixel_straightness":straightness_score(px_flat),
        "v1_curvature":      curvature_score(v1_feats),
        "v1_straightness":   straightness_score(v1_feats),
    }
    if Z is not None:
        m["latent_curvature"]    = curvature_score(Z)
        m["latent_straightness"] = straightness_score(Z)
    return m

# ── Compute all metrics ────────────────────────────────────────────────────────
all_metrics = {}
traj_store  = {}

for name, info in {**px_results, **ae_results}.items():
    Z_arg = info.get("Z")
    m = compute_metrics(info["frames"], clean_frames, Z=Z_arg)
    m["label"] = info.get("label", name)
    all_metrics[name] = m
    px_flat  = info["frames"].reshape(CONFIG["num_frames"], -1)
    v1_feats = get_v1_features(info["frames"])
    traj_store[name] = {
        "pixel":  pca_2d(px_flat),
        "v1":     pca_2d(v1_feats),
        "latent": pca_2d(Z_arg) if Z_arg is not None else None,
        "label":  info.get("label", name),
    }

# ── Print metrics table ────────────────────────────────────────────────────────
KEYS_ORD = (
    ["noisy_baseline", "clean_reference"] +
    [f"pixel_poly_{n}p" for n in CONFIG["poly_params"]] +
    ["ae_reconstruction"] +
    [f"ae_poly_{n}p" for n in CONFIG["poly_params"]]
)
print(f"{'Method':<30}{'PSNR':>7}{'SSIM':>7}{'V1-Curv':>10}{'V1-Str':>9}{'Px-Str':>9}")
print("─" * 72)
rows_json = []
for k in KEYS_ORD:
    m = all_metrics[k]
    print(f"{m['label']:<30}{m['psnr']:>7.2f}{m['temporal_ssim']:>7.4f}"
          f"{m['v1_curvature']:>10.4f}{m['v1_straightness']:>9.4f}"
          f"{m['pixel_straightness']:>9.4f}")
    rows_json.append({"method": k, **{kk: (float(vv) if not isinstance(vv, str) else vv)
                                      for kk, vv in m.items()}})

json_path = os.path.join(DIRS["tables"], "metrics.json")
with open(json_path, "w") as f:
    json.dump(rows_json, f, indent=2)
print(f"\nSaved: {json_path}")

# ══════════════════════════════════════════════════════════════════════════════
# PLOTS
POLY_KEYS_PX = [f"pixel_poly_{n}p" for n in CONFIG["poly_params"]]
POLY_KEYS_AE = [f"ae_poly_{n}p"    for n in CONFIG["poly_params"]]
LABELS_PX    = [all_metrics[k]["label"] for k in POLY_KEYS_PX]
LABELS_AE    = [all_metrics[k]["label"] for k in POLY_KEYS_AE]
DEGREES      = [n - 1 for n in CONFIG["poly_params"]]
COLORS       = plt.cm.tab10(np.linspace(0, 0.9, len(POLY_KEYS_PX)))

# ── Plot 1: Frame strip ────────────────────────────────────────────────────────
SHOW = [0, 4, 8, 12]
row_keys = (["noisy_baseline"] + POLY_KEYS_PX + ["ae_reconstruction"] + POLY_KEYS_AE)
row_src  = {**px_results, **ae_results}

fig, axes = plt.subplots(len(row_keys), len(SHOW),
                         figsize=(len(SHOW) * 2.5, len(row_keys) * 2.1))
plt.subplots_adjust(hspace=0.1, wspace=0.04)
for ri, rk in enumerate(row_keys):
    frames = row_src[rk]["frames"]
    lbl    = all_metrics[rk]["label"]
    for ci, fi in enumerate(SHOW):
        ax = axes[ri, ci]
        ax.imshow(np.clip(frames[fi], 0, 1))
        ax.set_xticks([]); ax.set_yticks([])
        if ci == 0:
            ax.set_ylabel(lbl, fontsize=7, rotation=0, labelpad=82, va="center")
        if ri == 0:
            ax.set_title(f"Frame {fi}", fontsize=8)
fig.suptitle("GIF Frame-Strip: Polynomial Degree Comparison", fontsize=11, y=1.01)
fig.savefig(os.path.join(DIRS["plots"], "frame_strip_comparison.png"),
            dpi=130, bbox_inches="tight")
plt.close(fig)
print("Saved: frame_strip_comparison.png")

# ── Plot 2: PSNR & SSIM ────────────────────────────────────────────────────────
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(len(DEGREES)); w = 0.35
for ax, metric, title, c_px, c_ae in [
    (axs[0], "psnr",         "PSNR ↑ vs Ground Truth (dB)", "#2a9d8f", "#e76f51"),
    (axs[1], "temporal_ssim","Temporal SSIM ↑",              "#264653", "#f4a261"),
]:
    ax.bar(x - w/2, [all_metrics[k][metric] for k in POLY_KEYS_PX], w,
           label="Pixel-Space", color=c_px, alpha=0.85)
    ax.bar(x + w/2, [all_metrics[k][metric] for k in POLY_KEYS_AE], w,
           label="AE Latent",   color=c_ae, alpha=0.85)
    ax.axhline(all_metrics["noisy_baseline"][metric],  ls="--", lw=1, color="gray",  label="Noisy")
    ax.axhline(all_metrics["clean_reference"][metric], ls=":",  lw=1, color="black", label="Clean")
    ax.set_xticks(x)
    ax.set_xticklabels([f"deg-{d}\n({d+1}p)" for d in DEGREES])
    ax.set_title(title, fontsize=11); ax.legend(fontsize=8)
fig.suptitle("Fidelity Metrics × Polynomial Degree × Space", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(DIRS["plots"], "psnr_ssim_comparison.png"),
            dpi=140, bbox_inches="tight")
plt.close(fig)
print("Saved: psnr_ssim_comparison.png")

# ── Plot 3: V1 Curvature & Straightness ───────────────────────────────────────
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric, title, c_px, c_ae in [
    (axs[0], "v1_curvature",    "V1 Curvature ↓ (rad)", "#457b9d", "#e9c46a"),
    (axs[1], "v1_straightness", "V1 Straightness ↑",    "#6a4c93", "#f4a261"),
]:
    ax.bar(x - w/2, [all_metrics[k][metric] for k in POLY_KEYS_PX], w,
           label="Pixel-Space", color=c_px, alpha=0.85)
    ax.bar(x + w/2, [all_metrics[k][metric] for k in POLY_KEYS_AE], w,
           label="AE Latent",   color=c_ae, alpha=0.85)
    ax.axhline(all_metrics["noisy_baseline"][metric],  ls="--", lw=1, color="gray",  label="Noisy")
    ax.axhline(all_metrics["clean_reference"][metric], ls=":",  lw=1, color="black", label="Clean")
    ax.set_xticks(x); ax.set_xticklabels([f"deg-{d}\n({d+1}p)" for d in DEGREES])
    ax.set_title(title, fontsize=11); ax.legend(fontsize=8)
fig.suptitle("PSH Metrics: V1 Perceptual-Space Trajectory Analysis", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(DIRS["plots"], "v1_curvature_straightness.png"),
            dpi=140, bbox_inches="tight")
plt.close(fig)
print("Saved: v1_curvature_straightness.png")

# ── Plot 4: Metric vs Degree (line) ───────────────────────────────────────────
fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (metric, ylabel, c_px, c_ae) in zip(axs, [
    ("v1_curvature",  "V1 Curvature ↓ (rad)",  "#457b9d", "#e9c46a"),
    ("pixel_curvature","Pixel Curvature ↓ (rad)","#2a9d8f","#e76f51"),
    ("temporal_ssim", "Temporal SSIM ↑",        "#6a4c93", "#f4a261"),
]):
    ax.plot(DEGREES, [all_metrics[k][metric] for k in POLY_KEYS_PX],
            "o-", color=c_px, lw=2.0, label="Pixel", ms=7)
    ax.plot(DEGREES, [all_metrics[k][metric] for k in POLY_KEYS_AE],
            "s-", color=c_ae, lw=2.0, label="AE Latent", ms=7)
    ax.axhline(all_metrics["noisy_baseline"][metric],  ls="--", lw=1.1, color="gray",  label="Noisy")
    ax.axhline(all_metrics["clean_reference"][metric], ls=":",  lw=1.1, color="black", label="Clean")
    ax.set_xticks(DEGREES); ax.set_xticklabels([f"deg-{d}\n({d+1}p)" for d in DEGREES])
    ax.set_title(ylabel, fontsize=10); ax.grid(True, alpha=0.3); ax.legend(fontsize=8)
fig.suptitle("Metric vs. Polynomial Degree: Pixel-Space vs AE Latent", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(DIRS["plots"], "metric_vs_degree.png"),
            dpi=140, bbox_inches="tight")
plt.close(fig)
print("Saved: metric_vs_degree.png")

# ── Plot 5: PCA Trajectory Plots ──────────────────────────────────────────────
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, field, title in zip(axs,
    ["pixel", "v1", "latent"],
    ["Pixel-Space PCA", "V1 Perceptual PCA", "AE Latent PCA"]):
    for bk, ls in [("noisy_baseline","--"), ("ae_reconstruction", "-.")]:
        if bk in traj_store and traj_store[bk][field] is not None:
            tr = traj_store[bk][field]
            ax.plot(tr[:,0], tr[:,1], ls=ls, color="gray", lw=1.3,
                    label=all_metrics[bk]["label"], alpha=0.7)
    for ki, k in enumerate(POLY_KEYS_PX):
        if traj_store[k][field] is None: continue
        tr = traj_store[k][field]
        ax.plot(tr[:,0], tr[:,1], "-", color=COLORS[ki], lw=1.8, alpha=0.9,
                label=f"Px deg-{DEGREES[ki]}")
        ax.scatter(tr[0,0], tr[0,1], marker="x", s=50, color=COLORS[ki])
    for ki, k in enumerate(POLY_KEYS_AE):
        if traj_store[k][field] is None: continue
        tr = traj_store[k][field]
        ax.plot(tr[:,0], tr[:,1], "--", color=COLORS[ki], lw=1.4, alpha=0.7,
                label=f"AE deg-{DEGREES[ki]}")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.grid(True, alpha=0.25); ax.legend(fontsize=6, ncol=2)
fig.suptitle("2D PCA Trajectory Comparison: All Methods × All Spaces", fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(DIRS["plots"], "pca_trajectories.png"),
            dpi=140, bbox_inches="tight")
plt.close(fig)
print("Saved: pca_trajectories.png")

# ── Plot 6: AE training loss ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(ae_losses, color="#e76f51", lw=1.8)
ax.set_xlabel("Epoch"); ax.set_ylabel("MSE Loss")
ax.set_title("AE Training Loss"); ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(DIRS["plots"], "ae_training_loss.png"),
            dpi=130, bbox_inches="tight")
plt.close(fig)
print("Saved: ae_training_loss.png")

# ── Plot 7: Perception–Distortion trade-off ────────────────────────────────────
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for ki, (k_px, k_ae) in enumerate(zip(POLY_KEYS_PX, POLY_KEYS_AE)):
    deg = DEGREES[ki]
    for k, mk, offset in [(k_px,"o",(6,4)), (k_ae,"s",(6,-12))]:
        ax.scatter(all_metrics[k]["v1_curvature"], all_metrics[k]["psnr"],
                   color=COLORS[ki], marker=mk, s=90, zorder=3)
        prefix = "Px" if k.startswith("pixel") else "AE"
        ax.annotate(f"{prefix}-{deg+1}p", (all_metrics[k]["v1_curvature"],
                    all_metrics[k]["psnr"]),
                    textcoords="offset points", xytext=offset, fontsize=8)
for bk, (mk, col) in [("noisy_baseline",("*","gray")),
                       ("clean_reference",("D","black"))]:
    ax.scatter(all_metrics[bk]["v1_curvature"], all_metrics[bk]["psnr"],
               marker=mk, s=130, color=col, zorder=5, label=all_metrics[bk]["label"])
# Legend proxy for poly degrees
for ki, deg in enumerate(DEGREES):
    ax.plot([], [], "o-", color=COLORS[ki], label=f"deg-{deg} ({deg+1}p)")
ax.set_xlabel("V1 Curvature ↓  (lower = perceptually smoother)", fontsize=10)
ax.set_ylabel("PSNR ↑ (dB) vs Ground Truth", fontsize=10)
ax.set_title("Perception–Distortion Trade-Off\n(○ = Pixel-space, □ = AE-Latent)", fontsize=11)
ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(os.path.join(DIRS["plots"], "perception_distortion_tradeoff.png"),
            dpi=140, bbox_inches="tight")
plt.close(fig)
print("Saved: perception_distortion_tradeoff.png")


In [ ]:
# ==========================================
# CELL 7 — Final Validation Check
# ==========================================
import traceback

errors = []

# 1. GIF files exist and are multi-frame
expected_gifs = (
    ["noisy_baseline.gif", "clean_reference.gif", "ae_reconstruction.gif"] +
    [f"pixel_poly_{n}p.gif" for n in CONFIG["poly_params"]] +
    [f"ae_poly_{n}p.gif"    for n in CONFIG["poly_params"]]
)
for g in expected_gifs:
    p = os.path.join(DIRS["gifs"], g)
    if not os.path.exists(p):
        errors.append(f"MISSING GIF: {g}")
    else:
        try:
            with Image.open(p) as im:
                if im.n_frames < 2:
                    errors.append(f"GIF has only 1 frame: {g}")
        except Exception as e:
            errors.append(f"Unreadable GIF {g}: {e}")

# 2. Frame dtype + value range
for name, info in {**px_results, **ae_results}.items():
    f = info["frames"]
    if f.dtype != np.float32:
        errors.append(f"Bad dtype in {name}: {f.dtype}")
    if not (0.0 <= f.min() and f.max() <= 1.0):
        errors.append(f"Out-of-range frames in {name}: [{f.min():.3f}, {f.max():.3f}]")
    expected_shape = (CONFIG["num_frames"], CONFIG["img_size"], CONFIG["img_size"], 3)
    if f.shape != expected_shape:
        errors.append(f"Wrong shape in {name}: {f.shape}")

# 3. Latent arrays: dtype + no NaN/Inf
for name, info in ae_results.items():
    Z = info.get("Z")
    if Z is not None:
        if Z.dtype != np.float32:
            errors.append(f"Bad latent dtype in {name}: {Z.dtype}")
        if np.isnan(Z).any() or np.isinf(Z).any():
            errors.append(f"NaN/Inf in latent {name}")

# 4. Metrics: no NaN/Inf
for name, m in all_metrics.items():
    for mk, mv in m.items():
        if isinstance(mv, float) and (np.isnan(mv) or np.isinf(mv)):
            errors.append(f"NaN/Inf in metrics[{name}][{mk}]")

# 5. Plots exist
expected_plots = [
    "frame_strip_comparison.png", "psnr_ssim_comparison.png",
    "v1_curvature_straightness.png", "metric_vs_degree.png",
    "pca_trajectories.png", "ae_training_loss.png",
    "perception_distortion_tradeoff.png",
]
for p in expected_plots:
    if not os.path.exists(os.path.join(DIRS["plots"], p)):
        errors.append(f"MISSING PLOT: {p}")

# 6. AE forward pass dtype + shape consistency
try:
    with torch.no_grad():
        dummy = torch.randn(2, 3, CONFIG["img_size"], CONFIG["img_size"]).to(DEV)
        out   = ae_model(dummy)
    if out.dtype != torch.float32:
        errors.append(f"AE output dtype: {out.dtype}")
    if tuple(out.shape) != tuple(dummy.shape):
        errors.append(f"AE output shape: {out.shape} != {dummy.shape}")
except Exception as e:
    errors.append(f"AE forward pass failed: {e}")

# 7. Polynomial regression numerical sanity (output must not exceed input noise floor)
for n in CONFIG["poly_params"]:
    s = poly_regression(noisy_frames, n)
    if s.dtype != np.float32:
        errors.append(f"poly_regression dtype wrong for n={n}: {s.dtype}")
    if s.shape != noisy_frames.shape:
        errors.append(f"poly_regression shape wrong for n={n}: {s.shape}")

# ── Report ─────────────────────────────────────────────────────────────────────
if errors:
    print("✗  Validation FAILED — issues found:")
    for e in errors:
        print(f"   • {e}")
else:
    print("✓  All validation checks passed.")
    print(f"   GIFs:    {len(expected_gifs)} files OK")
    print(f"   Plots:   {len(expected_plots)} files OK")
    print(f"   Frames:  dtype=float32, values in [0,1], shape correct")
    print(f"   Latents: dtype=float32, no NaN/Inf")
    print(f"   Metrics: no NaN/Inf across {len(all_metrics)} methods")
    print(f"   AE:      forward pass OK")
    print(f"   Poly:    regression output OK for all degrees")
    print(f"\nResults folder: {CONFIG['save_dir']}")
